# L3-10：Ascon release A/B 段

使用 Skill 的 artifact 与 overlay 检查加载正式 `pynq.bit`，读回 control/mailbox ID，
再通过现有真实 UART Transport 完成服务握手。

这里不重复完整 KAT；目标是证明通用 L3 流程能验证 Ascon release 的加载身份和服务身份。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import subprocess
import sys

import pynq


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


repo = Path("/home/xilinx/jupyter_notebooks/amd_fpga_2026").resolve()
assert (repo / "skill/ab_eval_pynq/test_board").is_dir(), (
    "fixed repository path does not exist: %s" % repo
)
os.chdir(str(repo))
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "skill/pynq-overlay-workflow/scripts"))

run_dir = (
    Path(os.environ["L3_RUN_DIR"]).resolve()  # 可选：不填时读取 l3_current_session.json
    if os.environ.get("L3_RUN_DIR")
    else Path(
        json.loads(
            (repo / "data/skill/l3_current_session.json").read_text(encoding="utf-8")
        )["run_dir"]
    ).resolve()
)
assert run_dir.is_dir(), run_dir
assert (repo / "data/skill").resolve() in run_dir.parents, (
    "L3_RUN_DIR must stay under data/skill"
)
assert (run_dir / "00_preflight.json").is_file(), "run l3_00_preflight.ipynb first"
run_id = run_dir.name


def base_record(case, claim_level):
    return {
        "case": case,
        "status": "RUNNING",
        "claim_level": claim_level,
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "run_id": run_id,
        "run_dir": str(run_dir),
        "hostname": platform.node(),
        "python_version": platform.python_version(),
        "pynq_version": pynq.__version__,
        "kernel": platform.platform(),
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
        ).strip(),
        "checks": [],
    }


def check(record, name, condition, detail=None):
    item = {"name": name, "status": "PASS" if condition else "FAIL", "detail": detail}
    record["checks"].append(item)
    assert condition, item

In [ ]:
from dataclasses import asdict
import time

from pynq.ps import Clocks
from pynq_skill import overlay_check
from src.pynq.pynq_transport import from_overlay

record = base_record("ascon_release", "hardware_measured")  # 只有真实板运行成功才保留此等级
record.update(
    {
        "simulated": None,  # 运行时由固件能力帧填写；禁止预先写 False
        "overlay_verification": None,  # 加载校验后自动填写
        "service_capabilities": None,  # 服务握手后自动填写
        "service_status": None,  # 服务握手后自动填写
    }
)
out_path = run_dir / "10_ascon_release.json"

try:
    release = repo / "board/vivado_zynq7000_xc7z020_pynq/release"
    overlay, info, verification = overlay_check.load_overlay(
        release / "pynq.bit",
        manifest=release / "manifest.json",
        required={
            "pynq_ctl_0": "xilinx.com:module_ref:pynq_ctl:1.0",
            "pynq_mbox_0": "xilinx.com:module_ref:pynq_mbox:1.0",
        },
        identity=[
            ("pynq_ctl_0", 0x08, 0x514E5950),
            ("pynq_ctl_0", 0x0C, 1),
            ("pynq_mbox_0", 0x1C, 0x4D424F58),
        ],
    )
    record["overlay_verification"] = verification
    record["fclk0_hwh_mhz"] = info["clocks"].get("fclk0_mhz")
    record["fclk0_pynq_register_derived_mhz"] = getattr(
        Clocks(), "fclk0_mhz", None
    )
    check(record, "overlay_loaded", overlay.is_loaded())
    check(record, "control_identity", verification["identity"][0]["value"] == 0x514E5950)
    check(record, "mailbox_identity", verification["identity"][2]["value"] == 0x4D424F58)

    transport = from_overlay(
        overlay,
        "pynq_ctl_0",
        "pynq_mbox_0",
        expected_firmware_id="pynq-svc-v1",
    )
    capabilities = transport.hello(time.monotonic() + 8.0)
    record["service_capabilities"] = asdict(capabilities)
    record["service_status"] = transport.status_snapshot()
    record["simulated"] = bool(capabilities.simulated)

    check(record, "service_not_simulated", not capabilities.simulated)
    check(record, "firmware_id", capabilities.firmware_id == "pynq-svc-v1")
    check(record, "capacity_16b", capabilities.max_message_bytes == 16 and capabilities.max_ad_bytes == 16)
    check(record, "sw_hw_ise_advertised", {int(mode) for mode in capabilities.modes} == {1, 2, 3})
    check(record, "firmware_selftest", transport.last_service["selftest_pass"] is True)
    check(record, "selftest_mask_7", transport.last_service["selftest_mask"] == 7)

    record["status"] = "PASS"
except BaseException as exc:
    record["status"] = "FAIL"
    record["error_type"] = type(exc).__name__
    record["error"] = str(exc)
    write_json(out_path, record)
    raise
else:
    write_json(out_path, record)
    print(json.dumps(record, indent=2, ensure_ascii=False))
    print("saved", out_path)